## 05_test_fl_flw_5_rounds.ipynb
## Validación del aprendizaje federado con múltiples rondas

Este notebook amplía el experimento FL-5 anterior incorporando múltiples rondas de comunicación, manteniendo la misma configuración federada y la misma estructura de clientes.

El objetivo es analizar el efecto acumulado de la agregación iterativa mediante FedAvg sobre el rendimiento del modelo y sobre las métricas de sostenibilidad. En lugar de realizar una única ronda de entrenamiento local seguida de una agregación, el proceso federado se repite durante varias rondas, permitiendo observar la evolución del modelo global a lo largo del tiempo.

La configuración experimental utilizada es la siguiente:

- 5 clientes federados.
- 5 rondas de comunicación.
- 5 épocas de entrenamiento local por cliente y por ronda.
- Agregación FedAvg al final de cada ronda.

CodeCarbon permanece integrado en cada cliente para recopilar métricas de sostenibilidad durante todo el proceso de entrenamiento. De esta forma, es posible obtener medidas tanto a nivel de cliente como del sistema completo, incluyendo consumo energético total, emisiones operativas y tiempo de ejecución acumulado en todas las rondas federadas.

Este notebook representa la primera implementación completa de un flujo de aprendizaje federado iterativo, proporcionando una aproximación más realista al comportamiento de los sistemas de aprendizaje federado y permitiendo estudiar la relación entre el número de rondas de comunicación, la precisión del modelo y las métricas de sostenibilidad.

0. Constantes e Imports

In [1]:
DATA_DIR = "../data"

DATASET = "CIFAR10"

SCENARIO = "federated"
NUM_CLIENTS = 5


BATCH_SIZE = 64

EPOCHS = 5

NUM_ROUNDS = 5

LEARNING_RATE = 0.001

RANDOM_SEED = 42


In [2]:
import torch

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import datasets
from torchvision import transforms

import numpy as np

import flwr as fl

from codecarbon import EmissionsTracker


1. Carga del Dataset y DataLoader

In [3]:

# ==================================================
# Transformaciones
# ==================================================

transform = transforms.ToTensor()

# ==================================================
# Dataset
# ==================================================

train_dataset = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

# ==================================================
# Partición FL-5
# ==================================================

client_size = len(train_dataset) // NUM_CLIENTS

client_datasets = random_split(
    train_dataset,
    [client_size] * NUM_CLIENTS
)

# ==================================================
# DataLoaders de clientes
# ==================================================

client_loaders = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    for ds in client_datasets
]

# ==================================================
# Test loader global
# ==================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

3. Modelo local para pruebas

In [5]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

4. Función de entrenamiento

In [6]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

5. Función de evaluación

In [7]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

6. Cliente Flower con set_parameters

In [ ]:
class FlowerClient(fl.client.NumPyClient):

    def __init__(
        self,
        model,
        train_loader,
        test_loader
    ):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader

    def get_parameters(
        self,
        config
    ):
        return [
            val.detach().numpy()
            for _, val in self.model.state_dict().items()
        ]
    
    def set_parameters(
        self,
        parameters
    ):
        
        params_dict = zip(
            self.model.state_dict().keys(),
            parameters
        )

        state_dict = {
            k: torch.tensor(v)
            for k, v in params_dict
        }

        self.model.load_state_dict(
            state_dict,
            strict=True
        )

    def fit(
        self,
        parameters,
        config
    ):
        self.set_parameters(parameters)

        local_criterion = nn.CrossEntropyLoss()

        local_optimizer = optim.Adam(
            self.model.parameters(),
            lr=LEARNING_RATE
        )

        tracker = EmissionsTracker(
            save_to_file=False
        )

        tracker.start()

        tracker.start_task("training")

        train_model(
            self.model,
            self.train_loader,
            local_optimizer,
            local_criterion,
            EPOCHS
        )

        training_data = tracker.stop_task()

        tracker.stop()

        metrics = {
            "energy_kwh": training_data.energy_consumed,
            "emissions_kg": training_data.emissions,
            "duration_s": training_data.duration
        }

        return (
            self.get_parameters(config),
            len(self.train_loader.dataset),
            metrics
        )
    
    def evaluate(
        self,
        parameters,
        config
    ):

        self.set_parameters(parameters)

        accuracy = evaluate_model(
          self.model,
          self.test_loader
        )

        return (
            0.0,
            len(self.test_loader.dataset),
            {
                "accuracy": accuracy
            }
        )

7. Función FedAvg

In [9]:
def fedavg(client_parameters):

    aggregated_parameters = []

    for layer_weights in zip(*client_parameters):

        aggregated_layer = np.mean(
            layer_weights,
            axis=0
        )

        aggregated_parameters.append(
            aggregated_layer
        )

    return aggregated_parameters

In [10]:
## Obtener pesos iniciales

global_model = SimpleCNN()

global_parameters = [
    val.detach().numpy()
    for _, val in global_model.state_dict().items()
]

In [11]:
# Crear clientes de Flower

clients = []

for i in range(NUM_CLIENTS):

    clients.append(
        FlowerClient(
            model=SimpleCNN(),
            train_loader=client_loaders[i],
            test_loader=test_loader
        )
    )

In [ ]:
# Entrenamiento federado de los 5 clientes durante 5 rondas

client_parameters = []

client_metrics = []

for round_num in range(NUM_ROUNDS):

    client_parameters = []

    client_metrics = []

    for client_id, client in enumerate(clients):

        updated_parameters, num_examples, metrics = client.fit(
            global_parameters,
            {}
        )
        metrics["round"] = round_num + 1
        metrics["client_id"] = client_id + 1
        print(
            f"Round {metrics['round']} | "
            f"Client {metrics['client_id']} | "
            f"Energy={metrics['energy_kwh']:.6f} kWh | "
            f"Emissions={metrics['emissions_kg']:.8f} kgCO2eq"
        )
        
        client_metrics.append(metrics)

        client_parameters.append(
            updated_parameters
        )
        
    global_parameters = fedavg(
        client_parameters
    )

[codecarbon WARNING @ 14:01:50] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 14:01:50] >>> Tracker's metadata:
[codecarbon INFO @ 14:01:50]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 14:01:50]   Python version: 3.12.3
[codecarbon INFO @ 14:01:50]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 14:01:50]   Available RAM : 7.632 GB
[codecarbon INFO @ 14:01:50]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 14:01:50]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 14:01:50]   GPU count: 0
[codecarbon INFO @ 14:01:50]   GPU model: 
[codecarbon INFO @ 14:01:50] Energy consumed for RAM : 0.000000 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 14:01:50] Delta energy consumed for CPU with cpu_load : 0.000000 kWh, power : 4.8000093312 W
[codecarbon INFO @ 14:01:50] Energy consumed for All CPU : 0.000000 kWh
[codecarbon INFO @ 14:01:50] 0.000000 kWh of electricity

Epoch 1/5 - Loss: 1.9357
Epoch 2/5 - Loss: 1.5144
Epoch 3/5 - Loss: 1.3838
Epoch 4/5 - Loss: 1.2781


[codecarbon INFO @ 14:02:20] Energy consumed for RAM : 0.000083 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 14:02:20] Delta energy consumed for CPU with cpu_load : 0.000079 kWh, power : 9.59274619824 W
[codecarbon INFO @ 14:02:20] Energy consumed for All CPU : 0.000079 kWh
[codecarbon INFO @ 14:02:20] 0.000162 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 14:02:20] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 14:02:20] >>> Tracker's metadata:
[codecarbon INFO @ 14:02:20]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 14:02:20]   Python version: 3.12.3
[codecarbon INFO @ 14:02:20]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 14:02:20]   Available RAM : 7.632 GB
[codecarbon INFO @ 14:02:20]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 14:02:20]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 14:02:20]  

Epoch 5/5 - Loss: 1.1999
Epoch 1/5 - Loss: 1.9304
Epoch 2/5 - Loss: 1.5343
Epoch 3/5 - Loss: 1.3926
Epoch 4/5 - Loss: 1.2934


[codecarbon INFO @ 14:02:50] Energy consumed for RAM : 0.000084 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 14:02:50] Delta energy consumed for CPU with cpu_load : 0.000081 kWh, power : 9.657688880825805 W
[codecarbon INFO @ 14:02:50] Energy consumed for All CPU : 0.000081 kWh
[codecarbon INFO @ 14:02:50] 0.000166 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 14:02:50] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 14:02:50] >>> Tracker's metadata:
[codecarbon INFO @ 14:02:50]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 14:02:50]   Python version: 3.12.3
[codecarbon INFO @ 14:02:50]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 14:02:50]   Available RAM : 7.632 GB
[codecarbon INFO @ 14:02:50]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 14:02:50]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 14:02:5

...

Epoch 5/5 - Loss: 0.1651
Epoch 1/5 - Loss: 0.6901
Epoch 2/5 - Loss: 0.4873
Epoch 3/5 - Loss: 0.3628
Epoch 4/5 - Loss: 0.2620


[codecarbon INFO @ 14:48:08] Energy consumed for RAM : 0.000083 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 14:48:08] Delta energy consumed for CPU with cpu_load : 0.000079 kWh, power : 9.571754007840001 W
[codecarbon INFO @ 14:48:08] Energy consumed for All CPU : 0.000079 kWh
[codecarbon INFO @ 14:48:08] 0.000162 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon WARNING @ 14:48:08] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 14:48:08] >>> Tracker's metadata:
[codecarbon INFO @ 14:48:08]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 14:48:08]   Python version: 3.12.3
[codecarbon INFO @ 14:48:08]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 14:48:08]   Available RAM : 7.632 GB
[codecarbon INFO @ 14:48:08]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 14:48:08]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 14:48:0

Epoch 5/5 - Loss: 0.1836
Epoch 1/5 - Loss: 0.6623
Epoch 2/5 - Loss: 0.4665
Epoch 3/5 - Loss: 0.3443
Epoch 4/5 - Loss: 0.2448


[codecarbon INFO @ 14:48:38] Energy consumed for RAM : 0.000081 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 14:48:38] Delta energy consumed for CPU with cpu_load : 0.000079 kWh, power : 9.6908360088 W
[codecarbon INFO @ 14:48:38] Energy consumed for All CPU : 0.000079 kWh
[codecarbon INFO @ 14:48:38] 0.000160 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 0.1703


8. Métricas federadas

In [15]:
energy_total = sum(
    m["energy_kwh"]
    for m in client_metrics
)

emissions_total = sum(
    m["emissions_kg"]
    for m in client_metrics
)

duration_total = sum(
    m["duration_s"]
    for m in client_metrics
)

print("Energy:", energy_total)
print("Emissions:", emissions_total)
print("Duration:", duration_total)

Energy: 0.0016746936515844941
Emissions: 0.0002914804300582812
Duration: 275.9779971480093


9. Crear modelo global

In [16]:
# Cargamos los pesos federados en el modelo global

global_model = SimpleCNN()

params_dict = zip(
    global_model.state_dict().keys(),
    global_parameters
)

state_dict = {
    k: torch.tensor(v)
    for k, v in params_dict
}

global_model.load_state_dict(
    state_dict,
    strict=True
)

<All keys matched successfully>

In [17]:
# Obtener accuracy del modelo global en el conjunto de prueba

global_accuracy = evaluate_model(
    global_model,
    test_loader
)

print(
    f"Global Accuracy: {global_accuracy:.4f}"
)

Global Accuracy: 0.7130
